## <font color='Black'>Construção de um Modelo de IA para Classificação de Dígitos Manuscritos (Ponta a Ponta)</font>🧠 **Contextos de aplicação:**🔹 Bancos - Leitura automática de valores em cheques.🔹 Correios - Reconhecimento de CEPs escritos à mão.🔹 Educação - Correção automática de provas com respostas numéricas.---

**Pipeline de um Projeto de IA:**Dados → Pré-processamento → Modelo → Treinamento → Validação → Teste → Inferência*O valor não está no dataset — está na capacidade de aplicar o mesmo pipeline em diferentes problemas!*Construir um modelo de Inteligência Artificial capaz de classificar imagens de dígitos manuscritos, considerando 10 categorias (dígitos de 0 a 9). Dada uma nova imagem de um dígito, o modelo deve ser capaz de classificar e indicar qual número está representado. <font color='red'>Uma imagem é uma matriz de pixels!</font>Este notebook adapta o pipeline construído para o CIFAR-10 (imagens coloridas de objetos) para o dataset **MNIST** (dígitos manuscritos em escala de cinza).---

## **1. Bibliotecas + Dados**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report

In [ ]:
# Carregar o MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Treino:", x_train.shape, y_train.shape)
print("Teste :", x_test.shape, y_test.shape)

In [ ]:
# Classes das imagens: os próprios dígitos
class_names = [str(i) for i in range(10)]

## **2. Visualização dos dados**

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Rótulo: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## **3. Pré-processamento**

In [ ]:
print("Antes:", x_train.min(), "a", x_train.max())

# Normalização: 0–255 → 0.0–1.0
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Depois:", x_train.min(), "a", x_train.max())

In [ ]:
# Reshape para (28, 28, 1) -> explicita o canal único (escala de cinza)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print("Formato treino:", x_train.shape)
print("Formato teste :", x_test.shape)

In [ ]:
# Guardamos os rótulos originais (0-9) para usar depois na matriz de confusão
# e no relatório de classificação, antes de transformar em One-Hot Encoding
y_train_original = y_train.copy()
y_test_original = y_test.copy()

# One-Hot Encoding: transforma o rótulo (ex: 7) em um vetor binário
# de 10 posições (ex: [0,0,0,0,0,0,0,1,0,0])
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

print("Exemplo antes :", y_train_original[0])
print("Exemplo depois:", y_train[0])

## **4. Construção do Modelo (CNN)**

In [ ]:
model = tf.keras.models.Sequential([

    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),

    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),

    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

## **5. Compilação e Treinamento**

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

## **6. Avaliação do Modelo**

In [ ]:
# Curva de acurácia
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia por época')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.show()

# Curva de perda (loss)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Perda (loss) por época')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Avaliação final no conjunto de teste
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print(f"\nPerda no teste   : {test_loss:.4f}")
print(f"Acurácia no teste: {test_accuracy:.2%}")

### **6.1 Matriz de Confusão e Relatório de Classificação**A **acurácia** mostra o desempenho geral do modelo.  A **matriz de confusão** permite analisar os acertos e erros de cada classe (cada dígito).- **Diagonal principal:** classificações corretas.- **Fora da diagonal:** dígitos que o modelo confundiu (ex: um 4 classificado como 9).

In [ ]:
# Previsões para todo o conjunto de teste
y_pred = model.predict(x_test, verbose=0)

# Classe com maior probabilidade
y_pred_classes = np.argmax(y_pred, axis=1)

# Matriz de confusão (usamos os rótulos originais, não o one-hot)
matriz_confusao = tf.math.confusion_matrix(
    y_test_original,
    y_pred_classes
)

# Exibição
plt.figure(figsize=(8, 7))
plt.imshow(matriz_confusao, cmap="Blues")
plt.title("Matriz de Confusão")
plt.xlabel("Classe Prevista")
plt.ylabel("Classe Real")
plt.xticks(range(10), class_names)
plt.yticks(range(10), class_names)
plt.colorbar()

for i in range(10):
    for j in range(10):
        plt.text(j, i, int(matriz_confusao[i, j]),
                 ha="center", va="center",
                 color="white" if matriz_confusao[i, j] > matriz_confusao.numpy().max() / 2 else "black")

plt.tight_layout()
plt.show()

In [ ]:
# Relatório de classificação (precisão, recall, f1-score por classe)
print(classification_report(y_test_original, y_pred_classes, target_names=class_names))

- Quais dígitos o modelo mais confundiu?- Por que esses dígitos podem ser visualmente semelhantes (ex: 4 e 9, 3 e 5)?

## **7. Inferência com uma imagem do teste**

In [ ]:
# Escolher uma imagem, alterando apenas o índice
indice = 0

imagem = x_test[indice]
classe_real = y_test_original[indice]

# Preparar para o modelo
imagem_modelo = np.expand_dims(imagem, axis=0)

# Fazer a previsão
previsoes = model.predict(imagem_modelo, verbose=0)

classe_prevista = np.argmax(previsoes)
confianca = np.max(previsoes)

# Mostrar resultado
plt.imshow(imagem.reshape(28, 28), cmap="gray")
plt.title(
    f"Real: {class_names[classe_real]} | "
    f"Previsto: {class_names[classe_prevista]} "
    f"({confianca:.1%})"
)
plt.axis("off")
plt.show()

### **7.1 Comparando várias previsões de uma vez**

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(15):
    imagem = x_test[i]
    classe_real = y_test_original[i]

    previsao = model.predict(np.expand_dims(imagem, axis=0), verbose=0)
    classe_prevista = np.argmax(previsao)

    plt.subplot(3, 5, i + 1)
    plt.imshow(imagem.reshape(28, 28), cmap="gray")
    cor = "green" if classe_prevista == classe_real else "red"
    plt.title(f"R:{classe_real} P:{classe_prevista}", color=cor)
    plt.axis("off")

plt.tight_layout()
plt.show()

## **8. Inferência com uma imagem externa (opcional)**

In [ ]:
# Carregar uma nova imagem sua (um dígito escrito à mão, fotografado)
img = tf.keras.utils.load_img(
    "/content/meu_digito.jpg",
    target_size=(28, 28),
    color_mode="grayscale"
)

# Preparar a imagem
img_array = tf.keras.utils.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Fazer a previsão
previsoes = model.predict(img_array, verbose=0)

classe_prevista = np.argmax(previsoes)
confianca = np.max(previsoes)

# Mostrar resultado
plt.imshow(img, cmap="gray")
plt.title(
    f"Previsto: {class_names[classe_prevista]} "
    f"({confianca:.1%})"
)
plt.axis("off")
plt.show()

⚠️ **Atenção:** o MNIST tem dígitos brancos sobre fundo preto, bem centralizados. Se sua imagem externa tiver fundo branco com dígito escuro, pode ser necessário inverter as cores antes de prever: `img_array = 1 - img_array`.

Se você chegou até aqui, parabéns! 🎆 🔥

**Agora explore outros datasets disponíveis no Keras, como: CIFAR-10, CIFAR-100 e Fashion-MNIST.** Observe como o mesmo pipeline pode ser adaptado para diferentes problemas.

Fim!